# 🔬 Caterva2 Dataset Exploration Agent (Chat UI + Plots)

A notebook-native chat interface for exploring Blosc2/Caterva2/HDF5 datasets.

---

### Quick start
1. **Run Cell 1** once to import the `Agent` and helpers.
2. **Run Cell 2** to create or reset the agent instance.
3. **Run Cell 3** to launch a scrollable chat UI with an input box at the bottom.
4. Ask questions continuously and scroll to compare previous interactions.

> **Requires:** A `.env` file at project root with `GROQ_API_KEY=...`


In [1]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
# Adds caterva2_agent/ to Python's module search path.
# Defines ask() — the notebook equivalent of main.py's interactive loop.
import sys, os

agent_dir = os.getcwd()
if agent_dir not in sys.path:
    sys.path.insert(0, agent_dir)

from agent import Agent

MAX_INPUT_CHARS = 5000  # same guard as main.py

def ask(message: str) -> None:
    """Send a message to the agent, mirroring all logic from main.py."""
    message = message.strip()
    if not message:
        print('[No input provided]')
        return
    if message.lower() in ['quit', 'exit']:
        print('Goodbye! Conversation and token counter reset.')
        agent.reset()
        return
    # reset calls agent.reset() directly — does NOT send the word to the LLM
    if message.lower() == 'reset':
        agent.reset()
        print('🔄 Conversation reset. Start a new question below.')
        return
    if len(message) > MAX_INPUT_CHARS:
        print(f'[Input too long: {len(message)} chars. Max is {MAX_INPUT_CHARS}]')
        return
    if len(message) > 2000:
        print(f'[Warning: Long input ({len(message)} chars) — may take longer]')
    try:
        response = agent.run(message)
        print(response)
    except Exception as e:
        print(f'[Error: {type(e).__name__}: {e}]')
        print("Try again or call ask('reset') to clear the conversation.")
        agent.reset()  # auto-reset on unhandled exception, same as main.py

print('Setup complete — ask() helper ready.')


Setup complete — ask() helper ready.


In [2]:
# ── Cell 2: Create Agent ───────────────────────────────────────────────────────
# Creates a new Agent instance with an empty conversation history.
# Re-run this cell any time you want to start a completely fresh session.
agent = Agent()
print("✅ Agent ready.")

✅ Agent ready.


In [3]:
# ── Cell 3: Launch chat UI (polished layout + artifacts) ────────────────────
import base64
import json
import ipywidgets as widgets
from IPython.display import HTML as DHTML, Image, Javascript, Markdown, display

ui_height = '74vh'

chat_output = widgets.Output(layout=widgets.Layout(
    border='1px solid #ddd',
    padding='10px',
    margin='0 0 8px 0',
    overflow='auto',
    flex='1 1 auto',
    min_height='260px'
))
chat_output.add_class('caterva-chat-scroll')

user_input = widgets.Textarea(
    placeholder='Write your message... (Enter to send, Option+Enter for newline)',
    layout=widgets.Layout(width='100%', height='64px', margin='0')
)
user_input.add_class('caterva-chat-input')

send_button = widgets.Button(description='Send', button_style='primary')
send_button.add_class('caterva-chat-send')
clear_button = widgets.Button(description='Clear chat UI')
reset_button = widgets.Button(description='Reset agent memory', button_style='warning')


def _scroll_chat_to_bottom() -> None:
    js = (
        "(function(){"
        "var wraps=Array.from(document.querySelectorAll('.caterva-chat-scroll'));"
        "if(!wraps.length){return;}"
        "var w=wraps[wraps.length-1];"
        "function targets(){return [w, w.querySelector('.jupyter-widgets-output-area'), w.querySelector('.widget-output'), w.querySelector('.jp-OutputArea-output')].filter(Boolean);}"
        "function go(){targets().forEach(function(t){t.scrollTop=t.scrollHeight;});}"
        "go();setTimeout(go,20);setTimeout(go,80);setTimeout(go,180);"
        "})();"
    )
    display(Javascript(js))


def _install_autoscroll_observer() -> None:
    # Keep the chat pinned to the latest content as outputs are appended asynchronously.
    js = (
        "(function(){"
        "var wraps=Array.from(document.querySelectorAll('.caterva-chat-scroll'));"
        "if(!wraps.length){return;}"
        "var w=wraps[wraps.length-1];"
        "if(w.dataset.catervaObserver==='1'){return;}"
        "w.dataset.catervaObserver='1';"
        "function targets(){return [w, w.querySelector('.jupyter-widgets-output-area'), w.querySelector('.widget-output'), w.querySelector('.jp-OutputArea-output')].filter(Boolean);}"
        "function go(){targets().forEach(function(t){t.scrollTop=t.scrollHeight;});}"
        "var obs=new MutationObserver(function(){go();setTimeout(go,30);setTimeout(go,120);});"
        "targets().forEach(function(t){obs.observe(t,{childList:true,subtree:true,characterData:true});});"
        "window.addEventListener('resize', go);"
        "go();"
        "})();"
    )
    display(Javascript(js))


def _autosize_input(change=None) -> None:
    text = user_input.value or ''
    line_count = max(2, min(10, text.count('\n') + 1))
    user_input.layout.height = f'{24 * line_count + 16}px'


def _bind_keyboard_shortcuts() -> None:
    js = "(function(){var wraps=Array.from(document.querySelectorAll('.caterva-chat-input'));var btns=Array.from(document.querySelectorAll('.caterva-chat-send'));if(!wraps.length||!btns.length){return;}var textarea=wraps[wraps.length-1].querySelector('textarea');var sendBtn=btns[btns.length-1];if(!textarea){return;}if(textarea.dataset.catervaBound==='1'){return;}textarea.dataset.catervaBound='1';textarea.addEventListener('keydown',function(e){if(e.key!=='Enter'){return;}if(e.altKey){return;}e.preventDefault();sendBtn.click();});})();"
    display(Javascript(js))


def _render_markdown_text(text: str) -> None:
    if text is None:
        return
    # Use IPython Markdown renderer so markdown syntax is formatted correctly.
    display(Markdown(str(text)))


def _render_artifact(artifact: dict) -> None:
    if not isinstance(artifact, dict):
        print(f"[Unsupported artifact: expected dict, got {type(artifact).__name__}]")
        return

    atype = artifact.get('type')

    if atype == 'plotly':
        try:
            import plotly.graph_objects as go
            fig_payload = artifact.get('figure')
            if isinstance(fig_payload, dict):
                display(go.Figure(fig_payload))
            else:
                print('[Plotly artifact missing valid figure dict]')
        except Exception as e:
            print(f"[Could not render Plotly artifact: {type(e).__name__}: {e}]")
        return

    if atype == 'image_base64':
        data_b64 = artifact.get('data')
        img_format = artifact.get('format', 'png')
        if not data_b64:
            print('[image_base64 artifact missing data]')
            return
        try:
            raw = base64.b64decode(data_b64)
            display(Image(data=raw, format=img_format))
        except Exception as e:
            print(f"[Could not decode image artifact: {type(e).__name__}: {e}]")
        return

    print(f"[Unsupported artifact type: {atype}]")


def _render_agent_payload(payload) -> None:
    if isinstance(payload, str):
        parsed = None
        try:
            parsed = json.loads(payload)
        except Exception:
            parsed = None
        if parsed is None:
            _render_markdown_text(payload)
            return
        payload = parsed

    if isinstance(payload, dict):
        text = payload.get('text') or payload.get('message') or payload.get('assistant')
        if text:
            _render_markdown_text(text)
        artifacts = payload.get('artifacts', [])
        if isinstance(artifacts, list):
            for artifact in artifacts:
                _render_artifact(artifact)
        else:
            print('[Invalid artifacts field: expected list]')
        return

    _render_markdown_text(str(payload))


def _start_interaction(user_message: str) -> None:
    with chat_output:
        display(DHTML("<hr style='border:0;border-top:2px solid #9aa0a6;margin:18px 0 14px 0;'>"))
        display(DHTML("<div style='font-weight:700; margin:0 0 6px 0;'>👩🏻‍💻 You:</div>"))
        display(DHTML(f"<div style='white-space:pre-wrap; margin:0 0 10px 0;'>{user_message}</div>"))
    _scroll_chat_to_bottom()


def _append_agent_response(content) -> None:
    with chat_output:
        display(DHTML("<div style='font-weight:700; margin:0 0 6px 0;'>֎  Agent:</div>"))
        _render_agent_payload(content)
        display(DHTML("<div style='height:10px;'></div>"))
    _scroll_chat_to_bottom()


def _append_system(text: str) -> None:
    with chat_output:
        display(DHTML(f"<div style='margin:0 0 8px 0;'><b>ⓘ System:</b> {text}</div>"))
    _scroll_chat_to_bottom()


def handle_message(message: str) -> None:
    message = (message or '').strip()
    if not message:
        return

    if message.lower() in {'quit', 'exit'}:
        _append_system('Session closed in UI (agent memory preserved).')
        return

    if message.lower() == 'reset':
        agent.reset()
        _append_system('Agent conversation memory reset.')
        return

    if len(message) > MAX_INPUT_CHARS:
        _append_system(f'Input too long: {len(message)} chars. Max is {MAX_INPUT_CHARS}.')
        return

    _start_interaction(message)
    try:
        response = agent.run(message)
        _append_agent_response(response)
    except Exception as e:
        _append_system(f'Error: {type(e).__name__}: {e}')
        _append_system("Try again or type 'reset' in the input box.")
        agent.reset()


def on_send(_):
    message = user_input.value
    user_input.value = ''
    _autosize_input()
    handle_message(message)


def on_clear(_):
    chat_output.clear_output()
    _append_system('Cleared visible chat history (agent memory still active).')


def on_reset(_):
    agent.reset()
    _append_system('Agent memory reset from button.')


send_button.on_click(on_send)
clear_button.on_click(on_clear)
reset_button.on_click(on_reset)
user_input.observe(_autosize_input, names='value')
_autosize_input()

input_label = widgets.HTML('<b>You:</b>', layout=widgets.Layout(margin='0 0 4px 0'))
buttons_row = widgets.HBox([send_button, clear_button, reset_button], layout=widgets.Layout(margin='6px 0 0 0'))
composer = widgets.VBox([input_label, user_input, buttons_row], layout=widgets.Layout(flex='0 0 auto', margin='0', padding='0'))

ui = widgets.VBox(
    [chat_output, composer],
    layout=widgets.Layout(
        height=ui_height,
        border='1px solid #ccc',
        padding='8px',
        overflow='hidden'
    ),
)

display(ui)
_bind_keyboard_shortcuts()
_install_autoscroll_observer()
_append_system('Chat UI ready. Press Enter to send. Use Option+Enter for a new line.')
_append_system('Rendering enabled: plain text + artifacts (plotly/image).')





<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>